In [ ]:
import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
from PermCell_Smooth import *
#from SHAPset import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
Run="Corrs"
import xgboost as xgb
import umap
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import pandas as pd
from POgSET import *
%matplotlib inline
import PyCytoData

In [ ]:
import ssl, urllib.request
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

with urllib.request.urlopen("https://example.com", context=ctx) as resp:
    print(resp.read()[:200])


In [ ]:
import os, ssl

# Disable HTTPS cert verification globally for this process
os.environ['PYTHONHTTPSVERIFY'] = '0'
ssl._create_default_https_context = ssl._create_unverified_context

import PyCytoData
PyCytoData.data.DataLoader().load_dataset("levine13", preprocess=True)


In [ ]:
DAT=PyCytoData.data.DataLoader().load_dataset("levine32",preprocess=True)

In [ ]:
CLM=[f.split("(")[0] for f in DAT.channels]

In [ ]:
CLM

In [ ]:
AD=ad.AnnData(pd.DataFrame(DAT.expression_matrix,columns=CLM),
              obs=pd.DataFrame(DAT.cell_types,columns=['Type']))

In [ ]:
AD=AD[AD.obs['Type']!='unassigned']

In [ ]:
AD

In [ ]:
CLM=['CD45RA', 'CD133', 'CD19',
       'CD22', 'CD11b', 'CD4', 'CD8', 'CD34', 'Flt3', 'CD20', 'CXCR4',
       'CD235ab', 'CD45', 'CD123', 'CD321', 'CD14', 'CD33', 'CD47', 'CD11c',
       'CD7', 'CD15', 'CD16', 'CD44', 'CD38', 'CD13', 'CD3', 'CD61', 'CD117',
       'CD49d', 'HLA-DR', 'CD64', 'CD41',]
AD=AD[:,CLM]

In [ ]:
def fast_umap_with_progress(
    adata,
    use_rep="X_pca",       # use your PCA here
    key_out="X_umap",
    n_neighbors=15,
    min_dist=0.3,
    n_components=2,
    metric="euclidean",
    n_epochs=120,          # fewer epochs = faster
    init="random",         # faster than 'spectral' for large N
    fit_subset=50_000,     # fit on this many cells, transform the rest
    random_state=1,
    verbose=True,
    **kwargs
):
    import numpy as np, umap
    X = adata.obsm[use_rep] if (use_rep and use_rep in adata.obsm) else adata.X
    N = X.shape[0]
    rng = np.random.RandomState(random_state)

    if (fit_subset is not None) and (N > fit_subset):
        idx_fit = rng.choice(N, fit_subset, replace=False)
        X_fit = X[idx_fit]
    else:
        idx_fit = None
        X_fit = X

    um = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric,
        n_epochs=n_epochs,
        init=init,
        random_state=random_state,
        low_memory=True,          # reduces RAM spikes
        verbose=verbose,          # progress bar
        **kwargs,
    )

    # Optional: pre-warm numba JIT (tiny run) to avoid big first-call lag
    _ = umap.UMAP(n_neighbors=5, n_epochs=10, random_state=0, verbose=False).fit_transform(X[:500])

    emb_fit = um.fit_transform(X_fit)  # progress bar appears here

    if idx_fit is None:
        adata.obsm[key_out] = emb_fit.astype("float32")
    else:
        # Fast interpolation (“transform”) for the rest of the cells
        emb_all = um.transform(X)
        adata.obsm[key_out] = emb_all.astype("float32")
        print(adata.obsm)
    return um

In [ ]:
print(AD.shape)
#sc.pp.pca(AD,n_comps=3)
fast_umap_with_progress(AD,use_rep=None,random_state=None)


In [ ]:
AD

In [ ]:
marker_sets_updown = {
    # --- Broad lineages ---
    "Tcell_core": {
        "up":   {"CD3", "CD7", "CD4", "CD8"},
        "down": {"CD14", "CD33", "CD11b", "CD19", "CD20", "CD22", "CD15", "CD16", "CD123"}
    },
    "Bcell_core": {
        "up":   {"CD19", "CD20", "CD22"},
        "down": {"CD3", "CD7", "CD14", "CD33", "CD11b", "CD11c", "CD15", "CD16"}
    },
    "NK_like": {  # proxy (no CD56 in panel)
        "up":   {"CD16", "CD7"},
        "down": {"CD3", "CD19", "CD20", "CD22", "CD14", "CD33"}
    },
    "Myeloid_core": {
        "up":   {"CD14", "CD33", "CD11b", "CD11c", "CD64", "HLA-DR"},
        "down": {"CD3", "CD7", "CD19", "CD20", "CD22"}
    },
    "Granulocyte_core": {
        "up":   {"CD15", "CD16", "CD11b"},
        "down": {"HLA-DR", "CD14", "CD3", "CD19", "CD20", "CD22"}
    },
    "DC_core": {
        "up":   {"CD11c", "HLA-DR"},
        "down": {"CD14", "CD15", "CD16", "CD3", "CD19", "CD20"}
    },

    # --- Monocyte flavors ---
    "Classical_Mono": {
        "up":   {"CD14", "CD64", "HLA-DR", "CD33", "CD11b"},
        "down": {"CD3", "CD19", "CD20", "CD22"}
    },
    "NonClassical_Mono": {  # proxy using CD16
        "up":   {"CD16", "CD11b", "CD33"},
        "down": {"CD14", "CD3", "CD19", "CD20", "CD22"}
    },

    # --- Progenitors / HSPC axis ---
    "HSC_like": {
        "up":   {"CD34", "CD133", "CD117", "CXCR4"},
        "down": {"CD38", "CD14", "CD33", "CD11b", "CD3", "CD19", "CD20", "CD22", "CD15", "CD16"}
    },
    "LMPP_like": {  # lymphoid-primed MPP
        "up":   {"CD34", "Flt3", "CD45RA"},
        "down": {"CD38", "CD14", "CD33", "CD11b", "CD15", "CD16"}
    },
    "CMP_GMP_like": {
        "up":   {"CD34", "CD117", "CD33"},
        "down": {"CD38"}  # light bias; tune to your data
    },

    # --- Erythroid / Mega ---
    "Erythroid": {
        "up":   {"CD235ab", "CD47"},
        "down": {"HLA-DR", "CD45", "CD14", "CD33"}
    },
    "Megakaryocyte_Platelet": {
        "up":   {"CD41", "CD61"},
        "down": {"CD45", "CD14", "CD33"}
    },

    # --- Naive/activation proxies ---
    "Naive_T": {
        "up":   {"CD3", "CD7", "CD45RA"},
        "down": {"CD14", "CD33", "CD11b"}
    },
    "Activated_T_proxy": {
        "up":   {"CD3", "CD7", "HLA-DR", "CD38"},
        "down": {"CD14", "CD33", "CD11b"}
    },

    # --- Trafficking / adhesion (cross-cutting) ---
    "Adhesion_Migration": {
        "up":   {"CD44", "CD49d", "CXCR4"},
        "down": set()
    },

    # --- Leukocyte sanity check ---
    "Leukocyte_common": {
        "up":   {"CD45"},
        "down": {"CD235ab"}  # exclude erythroid
    },
}


In [ ]:
%matplotlib inline

In [ ]:
sc.pl.umap(AD,color='Type')

In [ ]:
marker_sets_types = {
    # --- Granulocytes / innate ---
    "Basophils": {
        # With this panel, basophils are best separated as CD123+ HLA-DR−, CD16−, CD11c−, myeloid-core low
        "up":   {"CD123", "CD45RA"},
        "down": {"HLA-DR", "CD11c", "CD14", "CD15", "CD16", "CD33", "CD3", "CD19", "CD20", "CD22"}
    },

    # --- NK subsets (proxy without CD56) ---
    "CD16-_NK_cells": {
        "up":   {"CD7"},                       # CD7+ NK; CD16− subset lacks CD16
        "down": {"CD16", "CD3", "CD19", "CD20", "CD22", "CD14", "CD33"}
    },
    "CD16+_NK_cells": {
        "up":   {"CD16", "CD7"},
        "down": {"CD3", "CD19", "CD20", "CD22", "CD14", "CD33"}
    },

    # --- HSPC / HSC strata ---
    "CD34+CD38lo_HSCs": {
        "up":   {"CD34", "CD133", "CD117", "CXCR4"},
        "down": {"CD38", "Flt3", "CD3", "CD7", "CD19", "CD20", "CD22", "CD14", "CD33", "CD11b", "CD15", "CD16"}
    },
    "CD34+CD38+CD123-_HSPCs": {
        "up":   {"CD34", "CD38", "CD117", "CXCR4"},
        "down": {"CD123", "CD3", "CD7", "CD19", "CD20", "CD22", "CD14", "CD33", "CD11b", "CD15", "CD16"}
    },
    "CD34+CD38+CD123+_HSPCs": {
        "up":   {"CD34", "CD38", "CD117", "CD123", "Flt3", "CXCR4"},
        "down": {"CD3", "CD7", "CD19", "CD20", "CD22", "CD14", "CD33", "CD11b", "CD15", "CD16"}
    },

    # --- T cells ---
    "CD4_T_cells": {
        "up":   {"CD3", "CD4", "CD7"},
        "down": {"CD8", "CD14", "CD33", "CD11b", "CD19", "CD20", "CD22"}
    },
    "CD8_T_cells": {
        "up":   {"CD3", "CD8", "CD7"},
        "down": {"CD4", "CD14", "CD33", "CD11b", "CD19", "CD20", "CD22"}
    },

    # --- B-lineage ---
    "Mature_B_cells": {
        "up":   {"CD19", "CD20", "CD22", "HLA-DR"},
        "down": {"CD3", "CD7", "CD14", "CD33", "CD11b", "CD15", "CD16"}
    },
    "Pro_B_cells": {  # early B: CD34+, CD19+, CD22+, CD20–
        "up":   {"CD34", "CD19", "CD22", "CXCR4"},
        "down": {"CD20"}
    },
    "Pre_B_cells": {  # later than pro-B: CD34–, CD19+/CD22+, CD20 starts to rise
        "up":   {"CD19", "CD22", "CD38"},
        "down": {"CD34"}
    },
    "Plasma_B_cells": {  # plasma: CD38hi, CD19/20/22 low/negative in many contexts
        "up":   {"CD38"},
        "down": {"CD19", "CD20", "CD22"}
    },

    # --- Monocytes / DCs ---
    "Monocytes": {
        "up":   {"CD14", "CD33", "CD11b", "CD64", "HLA-DR"},
        "down": {"CD3", "CD7", "CD19", "CD20", "CD22"}
    },
    "pDCs": {
        "up":   {"CD123", "HLA-DR", "CD45RA"},
        "down": {"CD11c", "CD14", "CD15", "CD16", "CD3", "CD19", "CD20", "CD22"}
    },
}


In [ ]:
# Revised subsets only — replace these keys in your dict
marker_sets_types.update({
    "Basophils": {
        "up":   {"CD123", "CD45RA"},
        "down": {"HLA-DR", "CD11c", "CD16", "CD7", "CD3", "CD19", "CD20", "CD22", "CD14", "CD33"}
    },
    "CD16+_NK_cells": {
        "up":   {"CD16", "CD7", "CD45RA"},
        "down": {"CD3", "CD19", "CD20", "CD22", "CD14", "CD33", "CD11c", "CD123", "HLA-DR"}
    },
    "CD16-_NK_cells": {
        "up":   {"CD7", "CD45RA"},
        "down": {"CD16", "CD3", "CD19", "CD20", "CD22", "CD14", "CD33", "CD11c", "CD123", "HLA-DR"}
    },
    # (optional) Plasma tweak — make it less likely to beat NK spuriously
    "Plasma_B_cells": {
        "up":   {"CD38"},
        "down": {"CD19", "CD20", "CD22", "HLA-DR"}  # many plasma cells are HLA-DR low/neg
    },
})


In [ ]:
Z, P, Zabs, Pabs, Zdir = sipsic_like_scores_v3(
    AD, marker_sets_types,
    n_perm=1024,                   # or even 16 for smoke test
    prefer_permutation=True,     # <- important
    perm_batch=512,              # keeps RAM flat
    normalize_set_weights="l2",
    use_sparse_W=False,
    progress=True               # progress bars can add overhead in some envs
)

In [ ]:
B=list(Z.columns)
AD.obs[B]=Z[B].values

In [ ]:
AD=AD[AD.obs['Type']!='unassigned']

In [ ]:
sc.pl.umap(AD,color=B+['Type'],cmap='seismic',vcenter=0)

In [ ]:
# Suppose you have Z_dir as a DataFrame -> convert ONCE to arrays and lock order.
Z_arr = Z.to_numpy()            # (N,S)
P_arr = P.to_numpy() if 'P_df' in globals() else None
set_names = list(Z.columns)     # keep only for reporting/size_penalty (not used in math)



In [ ]:
import numpy as np
from typing import Dict, List, Optional, Tuple

def explain_domination_positional_v2(
    criteria: Dict[str, np.ndarray],    # each (N,S), SAME order for all
    maximize: Dict[str, bool],
    eps: Dict[str, float],
    cell_i: int,                        # 0-based cell index
    set_j: int,                         # 0-based set index
    drop_nan_sets: bool = True,
    set_names: Optional[List[str]] = None,
    top_k_print: int = 8
) -> Tuple[bool, List[int]]:
    """
    Explain whether (cell_i, set_j) is on front-0 under EXACTLY the same rules
    as poset_non_dominated_sets_per_cell_positional. Returns:
      (is_on_front0, dominators_idx_list) where indices are positional (0..S-1).
    """
    keys = list(criteria.keys())
    if not keys:
        raise ValueError("No criteria provided.")
    shapes = {k: criteria[k].shape for k in keys}
    if len(set(shapes.values())) != 1:
        raise ValueError(f"All criteria must share the same shape. Got: {shapes}")

    N, S = shapes[keys[0]]
    if not (0 <= cell_i < N) or not (0 <= set_j < S):
        raise IndexError(f"cell_i in [0,{N-1}], set_j in [0,{S-1}]")

    # Stack objectives for this cell: (S, K)
    M = np.stack([criteria[k][cell_i].astype(float, copy=False) for k in keys], axis=1)  # (S,K)
    K = M.shape[1]

    # Build direction/tolerance vectors (fill defaults for any missing keys)
    maximize_vec = np.array([bool(maximize.get(k, True)) for k in keys], dtype=bool)
    eps_vec = np.array([float(eps.get(k, 0.0)) for k in keys], dtype=float)

    # Validity mask exactly as in the poset fn
    valid = np.isfinite(M).all(axis=1) if drop_nan_sets else np.ones(S, dtype=bool)
    if not valid[set_j]:
        nm = str(set_j) if set_names is None else set_names[set_j]
        print(f"[!] Target set j={set_j} ({nm}) is invalid for this cell (NaN in criteria); "
              f"it is excluded from the poset and cannot be on the front.")
        return False, []

    # 1-D fast path matches poset behavior
    if K == 1:
        z = M[:, 0]
        if maximize_vec[0]:
            row_best = np.nanmax(z[valid])  # max among valid only
            is_front = (z[set_j] >= (row_best - eps_vec[0]))
        else:
            row_best = np.nanmin(z[valid])  # min among valid only
            is_front = (z[set_j] <= (row_best + eps_vec[0]))
        # dominators list for completeness in 1-D (any valid that strictly beats beyond eps)
        dominators = []
        if not is_front:
            if maximize_vec[0]:
                dominators = [j for j in np.where(valid)[0]
                              if z[j] > z[set_j] + eps_vec[0]]
            else:
                dominators = [j for j in np.where(valid)[0]
                              if z[j] < z[set_j] - eps_vec[0]]
        # Pretty print
        def _nm(j): return str(j) if set_names is None else str(set_names[j])
        if is_front:
            print(f"[OK] Cell {cell_i}, set {set_j} ({_nm(set_j)}) IS on front-0.")
        else:
            print(f"[X] Cell {cell_i}, set {set_j} ({_nm(set_j)}) is NOT on front-0.")
            print("Dominated by:", [(j, _nm(j)) for j in dominators])
        print("Target criteria:", {keys[0]: float(z[set_j])})
        return bool(is_front), dominators

    # General K>1 case — identical dominance test
    C = M.copy()
    C[:, maximize_vec] *= -1.0  # convert maximize -> minimize
    target = C[set_j]

    def _nm(j): return str(j) if set_names is None else str(set_names[j])

    dominators = []
    valid_idx = np.where(valid)[0]
    for j in valid_idx:
        if j == set_j:
            continue
        le_all = np.all(C[j] <= (target + eps_vec))
        lt_any = np.any(C[j] <  (target - eps_vec))
        if le_all and lt_any:
            dominators.append(j)

    is_front = (len(dominators) == 0)

    # Pretty print & top competitors by strength
    if is_front:
        print(f"[OK] Cell {cell_i}, set {set_j} ({_nm(set_j)}) IS on front-0.")
        print("Target criteria:", {k: float(criteria[k][cell_i, set_j]) for k in keys})
        return True, dominators

    print(f"[X] Cell {cell_i}, set {set_j} ({_nm(set_j)}) is NOT on front-0.")
    print("Target criteria:", {k: float(criteria[k][cell_i, set_j]) for k in keys})
    print("Dominated by:", [(j, _nm(j)) for j in dominators])

    # Order dominators by “strength” (sum of diffs in minimize-space)
    diffs = {j: float((C[j] - target).sum()) for j in dominators}
    for j, val in sorted(diffs.items(), key=lambda x: x[1])[:top_k_print]:
        vals = {k: float(criteria[k][cell_i, j]) for k in keys}
        print(f"  - j={j} ({_nm(j)}): total_diff={val:.4g}, vals={vals}")

    # Optional: also show top-by-z if available
    if "z" in criteria:
        order = np.argsort(-criteria["z"][cell_i, :])
        print("\nTop sets by z for this cell:")
        for j in order[:top_k_print]:
            print(f"  {j} ({_nm(j)}): z={criteria['z'][cell_i, j]:.4g}")

    return False, dominators


In [ ]:
import numpy as np
from typing import Dict, List, Tuple

def pack_criteria_positional(
    criteria: Dict[str, np.ndarray],   # each (N,S), SAME order for all
    maximize: Dict[str, bool],
    eps: Dict[str, float]
) -> Dict[str, object]:
    # Validate shapes & freeze order
    keys = list(criteria.keys())
    if not keys:
        raise ValueError("No criteria provided.")
    shapes = {k: criteria[k].shape for k in keys}
    if len(set(shapes.values())) != 1:
        raise ValueError(f"All criteria must share same shape. Got: {shapes}")
    N, S = shapes[keys[0]]
    # Build packed tensors in THIS fixed order of keys
    M = np.stack([criteria[k].astype(float, copy=False) for k in keys], axis=2)  # (N,S,K)
    maximize_vec = np.array([bool(maximize.get(k, True)) for k in keys], dtype=bool)
    eps_vec      = np.array([float(eps.get(k, 0.0))      for k in keys], dtype=float)
    valid = np.isfinite(M).all(axis=2)
    return {"M": M, "keys": tuple(keys), "maximize": maximize_vec, "eps": eps_vec, "valid": valid}

import numpy as np
from typing import Dict, List, Optional

def poset_from_pack_strict(
    pack: Dict[str, object],
    include_layers: bool = False,
    include_hasse: bool = False
):
    """
    Strict layer-peeling version that matches the explainer's dominance rule.
    - Computes front-0 by 'no incoming dominators'
    - Then peels layers by recomputing dominance among remaining sets
    """
    M     = pack["M"]            # (N,S,K), criteria (as floats)
    valid = pack["valid"]        # (N,S)   True if finite across criteria
    maxv  = pack["maximize"]     # (K,)    True -> maximize that criterion
    epsv  = pack["eps"]          # (K,)    per-criterion tolerance (>=0)
    N,S,K = M.shape

    ranks        = np.full((N,S), -1, np.int32)
    front0_mask  = np.zeros((N,S), dtype=bool)
    layers_idx   = [] if include_layers else None
    hasse_edges  = [] if include_hasse else None  # optional; can add later if needed

    # Precompute minimize-space values once
    C_all = M.copy()
    if K > 0:
        C_all[:, :, maxv] *= -1.0  # maximize -> minimize

    for i in range(N):
        valid_cols = np.where(valid[i])[0]
        if valid_cols.size == 0:
            if include_layers: layers_idx.append([])
            if include_hasse:  hasse_edges.append([])
            continue

        # Work on a dynamic "remaining" index list in this row
        remaining = valid_cols.tolist()
        layer_num = 0
        cell_layers = [] if include_layers else None

        while remaining:
            # Build Ms for remaining sets only
            Ms = C_all[i, remaining, :]              # (R, K)
            # Dominance within remaining
            le = (Ms[:, None, :] <= (Ms[None, :, :] + epsv[None, None, :])).all(axis=2)
            lt = (Ms[:, None, :] <  (Ms[None, :, :] - epsv[None, None, :])).any(axis=2)
            dom = le & lt
            np.fill_diagonal(dom, False)

            # A set is on the current front if it has NO incoming edges (no one dominates it)
            incoming = dom.any(axis=0)               # True if dominated by someone
            front_local = np.where(~incoming)[0].tolist()

            # Map to global set indices
            front_global = [remaining[idx] for idx in front_local]

            # Record ranks / front-0 mask
            ranks[i, front_global] = layer_num
            if layer_num == 0:
                front0_mask[i, front_global] = True
            if include_layers:
                cell_layers.append(front_global)

            # Remove this front from "remaining"
            remaining = [j for j in remaining if j not in front_global]
            layer_num += 1

        if include_layers:
            layers_idx.append(cell_layers)
        if include_hasse:
            # (Optional) One could compute Hasse edges using transitive reduction per layer;
            # omitted for brevity. Ask if you need it and I’ll add a consistent version.
            hasse_edges.append([])

    return {
        "ranks": ranks,
        "front0_mask": front0_mask,
        **({"layers_idx": layers_idx} if include_layers else {}),
        **({"hasse_edges": hasse_edges} if include_hasse else {}),
    }

def explain_from_pack(
    pack: Dict[str, object],
    cell_i: int,
    set_j: int,
    set_names: List[str] | None = None,
    top_k_print: int = 8
):
    M     = pack["M"]           # (N,S,K)
    valid = pack["valid"]       # (N,S)
    maxv  = pack["maximize"]    # (K,)
    epsv  = pack["eps"]         # (K,)
    N,S,K = M.shape
    if not (0 <= cell_i < N and 0 <= set_j < S):
        raise IndexError("cell_i or set_j out of range")

    if not valid[cell_i, set_j]:
        nm = str(set_j) if set_names is None else set_names[set_j]
        print(f"[!] Cell {cell_i}, set {set_j} ({nm}) is invalid (NaN in some criterion).")
        return False, []

    # 1-D fast path
    if K == 1:
        z = M[cell_i, :, 0]
        if maxv[0]:
            best = np.where(valid[cell_i], z, -np.inf).max()
            is_front = z[set_j] >= (best - epsv[0])
            dom = [j for j in range(S) if valid[cell_i, j] and (z[j] > z[set_j] + epsv[0])]
        else:
            best = np.where(valid[cell_i], z, +np.inf).min()
            is_front = z[set_j] <= (best + epsv[0])
            dom = [j for j in range(S) if valid[cell_i, j] and (z[j] < z[set_j] - epsv[0])]
    else:
        C = M.copy()
        C[:, :, maxv] *= -1.0
        target = C[cell_i, set_j, :]
        dom = []
        for j in range(S):
            if j == set_j or not valid[cell_i, j]:
                continue
            le_all = np.all(C[cell_i, j, :] <= (target + epsv))
            lt_any = np.any(C[cell_i, j, :] <  (target - epsv))
            if le_all and lt_any:
                dom.append(j)
        is_front = (len(dom) == 0)

    def _nm(j): return str(j) if set_names is None else set_names[j]
    vals = {f"k={t}": float(M[cell_i, set_j, t]) for t in range(K)}
    print(("[OK]" if is_front else "[X]"),
          f"Cell {cell_i}, set {set_j} ({_nm(set_j)})",
          "IS on front-0." if is_front else "is NOT on front-0.")
    print("Target criteria:", vals)
    if not is_front:
        print("Dominated by:", [(j, _nm(j)) for j in dom])
        # Strength ordering
        if K > 1:
            C = M.copy(); C[:, :, maxv] *= -1.0
            target = C[cell_i, set_j, :]
            diffs = {j: float((C[cell_i, j, :] - target).sum()) for j in dom}
            for j, val in sorted(diffs.items(), key=lambda x: x[1])[:top_k_print]:
                print(f"  - j={j} ({_nm(j)}): total_diff={val:.4g}, vals={ {f'k={t}': float(M[cell_i,j,t]) for t in range(K)} }")
    # Top by first criterion (usually z)
    order = np.argsort(-M[cell_i, :, 0])
    print("\nTop sets by criterion[0]:", [(_nm(j), float(M[cell_i, j, 0])) for j in order[:top_k_print]])
    return bool(is_front), dom


In [ ]:
Z_arr=Z.values

In [ ]:
# 1) Build your criteria ARRAYS with the SAME (N,S) order everywhere.
crit_arrays = {
    # If you want z-only:
    # "z": Z_arr,
    # Or the five you printed:
    "z": Z_arr,
    "absz": np.abs(Z_arr),
    "margin": build_poset_criteria_1to8_positional(Z_arr)["margin"],
    "coverage_penalty": build_poset_criteria_1to8_positional(Z_arr)["coverage_penalty"],
    "size_penalty": build_poset_criteria_1to8_positional(Z_arr, set_names=set_names, marker_sets=marker_sets_types)["size_penalty"],
}

# 2) Directions & tolerances for *exactly those keys*
maximize = {"z": True, "absz": True, "margin": True, "coverage_penalty": False, "size_penalty": False}
eps      = {"z": 0.25, "absz": 0.1, "margin": 0.1, "coverage_penalty": 0.0, "size_penalty": 0.0}


# 3) Pack once
pack = pack_criteria_positional(crit_arrays, maximize, eps)

# 4) Compute poset from the same pack
res = poset_from_pack_strict(pack)
front0_mask = res["front0_mask"]     # (N,S) bool

# # 5) Explain using the same pack (so it cannot disagree)
# i = 946
# j = 1   # e.g., CD16-_NK_cells column index in *your* fixed order
# explain_from_pack(pack, cell_i=i, set_j=j, set_names=set_names)


In [ ]:
sc.pp.pca(AD)
sc.pp.neighbors(AD)

In [ ]:
AD

In [ ]:
# --- Build ALL criteria once (positional) ---
# Z_arr: (N,S) z or Z_dir as numpy array
# P_arr: (N,S) p-values array or None
# set_names: list[str] length S (only used for size_penalty titles)
# marker_sets_types: your up/down dict (only for size_penalty)
# knn_conn: (N,N) sparse connectivities (e.g., adata.obsp["connectivities"]) or None

crit_all = build_poset_criteria_1to8_positional(
    Z=Z.values,
    P=P.values,                                # can be None
    set_names=set_names,                    # needed only if marker_sets not None
    marker_sets=marker_sets_types,          # for size_penalty
    knn_connectivities=AD.obsp['connectivities'],            # can be None
    coverage_threshold=0.25
)

# --- Choose which criteria to use (add/remove keys here) ---
keys_to_use = [
    "z", "absz", "margin",                  # strength & separation
    "neglogp", "neglogq",                   # significance (if P_arr provided)
    "knn_mean_z", "knn_var_z",              # neighborhood consistency (if knn provided)
    "coverage_penalty", "size_penalty"      # specificity & parsimony
]
# Keep only those that were actually computed:
crit_arrays = {k: crit_all[k] for k in keys_to_use if k in crit_all}

# --- Directions & eps for exactly these keys ---
base_maximize = {
    "z": True, "absz": True, "margin": True,
    "neglogp": True, "neglogq": True,
    "knn_mean_z": True,
    "knn_var_z": False,
    "coverage_penalty": False,
    "size_penalty": False,
}
base_eps = {
    # gentle slacks to avoid spurious domination
    "z": 0.05, "absz": 0.10, "margin": 0.10,
    "neglogp": 0.10, "neglogq": 0.10,
    "knn_mean_z": 0.10,
    "knn_var_z": 0.02,           # tiny; we're minimizing variance
    "coverage_penalty": 0.00,    # strict by default
    "size_penalty": 0.00,        # strict by default
}
# Trim to present keys:
maximize = {k: base_maximize[k] for k in crit_arrays.keys()}
eps      = {k: base_eps.get(k, 0.0) for k in crit_arrays.keys()}

# --- Pack once (locks tensors, order, maximize, eps) ---
pack = pack_criteria_positional(crit_arrays, maximize, eps)

# --- Compute POSET from the same pack (strict layer peeling; matches explainer) ---
res = poset_from_pack_strict(pack)
front0_mask = res["front0_mask"]     # (N,S) bool

# --- Explain using the SAME pack (cannot disagree) ---
i = 946
j = 1   # your CD16-_NK_cells column index in the fixed set order
explain_from_pack(pack, cell_i=i, set_j=j, set_names=set_names)


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Optional

def plot_pareto_front_grid_positional(
    XY: np.ndarray,                 # (N,2) embedding (e.g., adata.obsm["X_umap"])
    front0_mask: np.ndarray,        # (N,S) bool; True if set j is on front-0 for cell i
    set_names: Optional[List[str]] = None,
    sets_to_plot: Optional[List[int]] = None,  # list of column indices to plot; None = all
    ncols: int = 4,
    s_bg: float = 4.0,
    s_fg: float = 7.0,
    alpha_bg: float = 0.15,
    figsize_per_col: float = 4.0,
    title_prefix: str = ""
):
    """
    Purely positional plotting. One panel per set; highlighted cells are those
    where that set is on their Pareto front (front-0).

    - XY must be (N,2).
    - front0_mask must be (N,S) with the SAME row order as XY.
    - If set_names is provided, titles use them; otherwise, indices are shown.
    - sets_to_plot lets you subset columns by index.
    """
    XY = np.asarray(XY, dtype=float)
    if XY.ndim != 2 or XY.shape[1] != 2:
        raise ValueError(f"XY must be (N,2); got {XY.shape}")
    N = XY.shape[0]

    F = np.asarray(front0_mask, dtype=bool)
    if F.ndim != 2 or F.shape[0] != N:
        raise ValueError(f"front0_mask must be (N,S) with N={N}; got {F.shape}")
    S = F.shape[1]

    cols_idx = np.arange(S) if sets_to_plot is None else np.asarray(sets_to_plot, dtype=int)
    if cols_idx.size == 0:
        raise ValueError("No sets to plot (sets_to_plot is empty).")

    # titles
    if set_names is None:
        titles = [f"set {j}" for j in cols_idx]
    else:
        if len(set_names) != S:
            raise ValueError(f"set_names must have length S={S}; got {len(set_names)}")
        titles = [str(set_names[j]) for j in cols_idx]

    nS = cols_idx.size
    nrows = math.ceil(nS / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per_col*ncols, figsize_per_col*nrows), squeeze=False)
    axes = axes.ravel()

    # color cycle (matplotlib default)
    colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["tab:blue"])

    for i, j in enumerate(cols_idx):
        ax = axes[i]
        ax.scatter(XY[:, 0], XY[:, 1], s=s_bg, c="lightgray", alpha=alpha_bg, linewidths=0)
        mask = F[:, j]
        ax.scatter(XY[mask, 0], XY[mask, 1], s=s_fg, c=colors[i % len(colors)], alpha=0.95, linewidths=0)
        ax.set_title(f"{title_prefix}{titles[i]}  (n={int(mask.sum())})")
        ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)

    # hide unused panels
    for k in range(nS, len(axes)):
        axes[k].axis("off")

    plt.tight_layout()
    return fig


In [ ]:
M=AD.obs['Type'].str.contains('16')

In [ ]:
fig = plot_pareto_front_grid_positional(
    AD.obsm['X_umap'], res['front0_mask'], set_names=set_names,
    sets_to_plot=None, ncols=5, s_bg=3.5, s_fg=7.0, alpha_bg=0.12
)

sc.pl.umap(AD,color='Type')

In [ ]:
import numpy as np
from typing import Dict, Optional

def _domination_matrix_minspace(C: np.ndarray, eps: np.ndarray) -> np.ndarray:
    """
    C: (S,K) already in minimize-space (smaller is better).
    eps: (K,)
    Returns: dom (S,S) bool where dom[i,j] is True iff i dominates j.
    """
    # i <= j (within eps) for all k
    le = (C[:, None, :] <= (C[None, :, :] + eps[None, None, :])).all(axis=2)
    # i <  j (beyond eps) for some k
    lt = (C[:, None, :] <  (C[None, :, :] - eps[None, None, :])).any(axis=2)
    dom = le & lt
    np.fill_diagonal(dom, False)
    return dom

def poset_front0_for_cell_from_pack_fixed(pack: Dict[str, object], i: int) -> np.ndarray:
    """
    Exact front-0 for a single cell i using the strict definition above.
    """
    M     = pack["M"]        # (N,S,K)
    valid = pack["valid"]    # (N,S)
    maxv  = pack["maximize"] # (K,)
    epsv  = pack["eps"]      # (K,)
    N,S,K = M.shape

    mask_valid = valid[i]
    front = np.zeros(S, dtype=bool)
    if not mask_valid.any():
        return front

    # 1-D fast path
    if K == 1:
        z = M[i, :, 0]
        if maxv[0]:
            best = np.where(mask_valid, z, -np.inf).max()
            front = (z >= (best - epsv[0])) & mask_valid
        else:
            best = np.where(mask_valid, z, +np.inf).min()
            front = (z <= (best + epsv[0])) & mask_valid
        return front

    # K>1: convert to minimize-space
    C = M[i, :, :].copy()
    C[:, maxv] *= -1.0
    C = C[mask_valid]  # (S',K) only valid sets

    dom = _domination_matrix_minspace(C, epsv)
    incoming = dom.any(axis=0)        # True if dominated by someone
    front_local = ~incoming           # no incoming edges
    front[np.where(mask_valid)[0][front_local]] = True
    return front

def poset_from_pack_fixed(
    pack: Dict[str, object],
    include_layers: bool = False
) -> Dict[str, np.ndarray]:
    """
    Strict layer-peeling POSET (matches the explainer logic).
    """
    M     = pack["M"]        # (N,S,K)
    valid = pack["valid"]    # (N,S)
    maxv  = pack["maximize"] # (K,)
    epsv  = pack["eps"]      # (K,)
    N,S,K = M.shape

    ranks = np.full((N, S), -1, np.int32)
    front0_mask = np.zeros((N, S), dtype=bool)
    layers_idx = [] if include_layers else None

    # Precompute minimize-space
    C_all = M.copy()
    if K > 1:
        C_all[:, :, maxv] *= -1.0

    for i in range(N):
        valid_cols = np.where(valid[i])[0]
        if valid_cols.size == 0:
            if include_layers: layers_idx.append([])
            continue

        remaining = valid_cols.tolist()
        layer = 0
        cell_layers = [] if include_layers else None

        if K == 1:
            # 1-D special case is closed form
            z = M[i, remaining, 0]
            if maxv[0]:
                best = z.max()
                front_local = np.where(z >= (best - epsv[0]))[0].tolist()
            else:
                best = z.min()
                front_local = np.where(z <= (best + epsv[0]))[0].tolist()
            front_global = [remaining[t] for t in front_local]
            ranks[i, front_global] = 0
            front0_mask[i, front_global] = True
            if include_layers:
                cell_layers.append(front_global)
            if include_layers: layers_idx.append(cell_layers)
            continue

        # K>1: peel fronts by recomputing dominance among remaining
        while remaining:
            Ms = C_all[i, remaining, :]       # (R,K) in minimize-space
            dom = _domination_matrix_minspace(Ms, epsv)
            incoming = dom.any(axis=0)
            front_local = np.where(~incoming)[0].tolist()
            front_global = [remaining[t] for t in front_local]

            ranks[i, front_global] = layer
            if layer == 0:
                front0_mask[i, front_global] = True
            if include_layers:
                cell_layers.append(front_global)

            # drop current front and continue
            remaining = [j for j in remaining if j not in front_global]
            layer += 1

        if include_layers:
            layers_idx.append(cell_layers)

    out = {"ranks": ranks, "front0_mask": front0_mask}
    if include_layers: out["layers_idx"] = layers_idx
    return out


In [ ]:
def assert_front0_matches_explainer(pack, res, cell_i):
    f_poset = res["front0_mask"][cell_i]
    f_expl  = poset_front0_for_cell_from_pack_fixed(pack, cell_i)
    if not np.array_equal(f_poset, f_expl):
        diff = np.where(f_poset ^ f_expl)[0]
        raise AssertionError(f"Front-0 mismatch at cell {cell_i}; differing set indices: {diff}")

def assert_1d_special_case(Z_arr):
    # z-only pack → front equals argmax within eps
    pack = pack_criteria_positional({"z": Z_arr}, {"z": True}, {"z": 0.0})
    res  = poset_from_pack_fixed(pack)
    front_simple = (Z_arr == Z_arr.max(axis=1, keepdims=True))
    if not np.array_equal(res["front0_mask"], front_simple):
        raise AssertionError("1D z-only check failed.")


In [ ]:
Z_arr=Z.values
P_arr=P.values
knn_conn=AD.obsp['connectivities']

In [ ]:
# 1) Build criteria (same as you do now)
crit_all = build_poset_criteria_1to8_positional(
    Z=Z_arr, P=P_arr,
    set_names=set_names, marker_sets=marker_sets_types,
    knn_connectivities=knn_conn, coverage_threshold=0.25
)
keys_to_use = [k for k in ["z","absz","margin","neglogp","neglogq","knn_mean_z","knn_var_z","coverage_penalty","size_penalty"] if k in crit_all]
crit_arrays = {k: crit_all[k] for k in keys_to_use}

# 2) Directions & eps (make sure every key in crit_arrays has an entry)
maximize = {k: True  for k in crit_arrays}
for k in ["knn_var_z","coverage_penalty","size_penalty"]: 
    if k in maximize: maximize[k] = False
eps = {k: 0.0 for k in crit_arrays}
eps.update({k:v for k,v in {"z":0.001,"absz":0.10,"margin":0.10,"neglogp":0.10,"neglogq":0.10,"knn_mean_z":0.10,"knn_var_z":0.02}.items() if k in crit_arrays})

# 3) Pack and run with the FIXED poset
pack = pack_criteria_positional(crit_arrays, maximize, eps)
res  = poset_from_pack_fixed(pack)
front0_mask = res["front0_mask"]

# 4) Sanity: explainer/poset must agree (e.g., cell 946)
assert_front0_matches_explainer(pack, res, cell_i=946)

# (Optional) 1D sanity when you test with z-only
# assert_1d_special_case(Z_arr)


In [ ]:
Z.columns[front0_mask[946]]

In [ ]:
crit_arrays = {
    "z": Z_arr, 'absz':np.abs(Z_arr),
    "margin": build_poset_criteria_1to8_positional(Z_arr)["margin"],
}
maximize = {"z": True, "margin": True,"absz":True}
eps      = {"z": 0.2, "margin": .05,"absz":0.}   # try 0.20/0.05 if still too many



In [ ]:
eps

In [ ]:
pack = pack_criteria_positional(crit_arrays, maximize, eps)
res  = poset_from_pack_fixed(pack)        # the strict version you installed
front0_mask = res["front0_mask"]

In [ ]:
fig = plot_pareto_front_grid_positional(
    AD.obsm['X_umap'], res['front0_mask'], set_names=set_names,
    sets_to_plot=None, ncols=5, s_bg=3, s_fg=3.0, alpha_bg=0.12
)

sc.pl.umap(AD,color='Type')

In [ ]:
front_counts = front0_mask.sum(axis=1)  # (N,)
print("front0 labels per cell — mean:", front_counts.mean(), "median:", np.median(front_counts))


In [ ]:
sns.histplot(front_counts)

In [ ]:
(front_counts>1).sum()

In [ ]:
S = front0_mask.shape[1]
A = front0_mask.astype(np.int32)      # (N, S)
cofront = (A.T @ A)                   # (S, S) integer counts
np.fill_diagonal(cofront, 0)

# Top conflicting pairs (by co-front count):
pairs = []
for a in range(S):
    for b in range(a+1, S):
        pairs.append(((a,b), cofront[a,b]))
pairs = sorted(pairs, key=lambda x: -x[1])[:]
print("Top co-front pairs (idx, count):")
#print([(set_names[i], set_names[j], c) for (i,j), c in pairs])

from tabulate import tabulate

rows = [(set_names[i], set_names[j], c) for (i, j), c in pairs]
print(tabulate(rows, headers=["Set A", "Set B", "Count"], tablefmt="github"))    
# If you have set_names:
# print([(set_names[i], set_names[j], c) for (i,j), c in pairs])


In [ ]:
for i,S in enumerate(set_names):
    print(f"{i} {S}")

In [ ]:
M=front_counts==1
print(M.sum())

In [ ]:
fig = plot_pareto_front_grid_positional(
    AD[M].obsm['X_umap'], res['front0_mask'][M], set_names=set_names,
    sets_to_plot=None, ncols=5, s_bg=3, s_fg=3.0, alpha_bg=0.12
)

sc.pl.umap(AD[M],color='Type')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

def confusion_single_front_vs_labels(
    front0_mask: np.ndarray,         # (N,S) bool
    set_names: list,                 # length S
    true_labels: np.ndarray,         # length N, strings or pandas Series
    drop_unseen_labels: bool = True, # drop classes that never appear in preds/true
    normalize: str | None = None     # None | 'true' | 'pred' | 'all' (sklearn style)
):
    """
    Returns (conf_df, report_str, idx_used) where:
      - conf_df: pandas.DataFrame confusion matrix (rows=true, cols=pred)
      - report_str: sklearn classification_report for included cells/classes
      - idx_used: np.ndarray of cell indices used (single-front & non-null label)
    """
    N, S = front0_mask.shape
    if len(set_names) != S:
        raise ValueError(f"set_names length {len(set_names)} != S {S}")
    if len(true_labels) != N:
        raise ValueError(f"true_labels length {len(true_labels)} != N {N}")

    # 1) pick cells with exactly one front set
    single = (front0_mask.sum(axis=1) == 1)
    if not np.any(single):
        raise ValueError("No cells with exactly one front-0 set.")

    # 2) predicted label = that unique set's name
    pred_idx = front0_mask.argmax(axis=1)            # index of True in each row
    y_pred = np.array(set_names, dtype=object)[pred_idx]

    # 3) filter to single-front cells and non-null true labels
    y_true = np.asarray(true_labels, dtype=object)
    ok = single & pd.notna(y_true)
    y_true = y_true[ok]
    y_pred = y_pred[ok]
    idx_used = np.flatnonzero(ok)

    # 4) optionally drop labels that never appear in either y_true or y_pred
    if drop_unseen_labels:
        classes = sorted(set(y_true) | set(y_pred))
    else:
        # keep full set of names in fixed order (true side might include labels not in set_names)
        classes = sorted(set(y_true) | set(y_pred))

    # 5) confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=classes, normalize=normalize)
    conf_df = pd.DataFrame(cm, index=pd.Index(classes, name="True"), columns=pd.Index(classes, name="Pred"))

    # 6) classification report (macro/weighted F1, per-class precision/recall)
    report_str = classification_report(y_true, y_pred, labels=classes, zero_division=0)

    return conf_df, report_str, idx_used


In [ ]:
conf_df, report, used = confusion_single_front_vs_labels(
    front0_mask=front0_mask,
    set_names=set_names,
    true_labels=AD.obs['Type'],
    normalize=None  # or 'true' to get per-row recall, 'pred' for precision, 'all' overall
)


In [ ]:
print(conf_df)      # raw counts (or normalized if chosen)
print()
print(report)       # precision/recall/F1 per class

# If you want a quick heatmap:
import matplotlib.pyplot as plt
plt.figure(figsize=(0.35*conf_df.shape[1]+3, 0.35*conf_df.shape[0]+3))
plt.imshow(conf_df.values, aspect="auto")
plt.colorbar()
plt.xticks(range(conf_df.shape[1]), conf_df.columns, rotation=90)
plt.yticks(range(conf_df.shape[0]), conf_df.index)
plt.title("Confusion matrix (single-front cells)")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

def preds_from_single_front(front0_mask: np.ndarray, set_names: list[str]) -> np.ndarray:
    idx = front0_mask.argmax(axis=1)
    return np.array(set_names, dtype=object)[idx]

def confusion_panels_single_front(
    front0_mask: np.ndarray,       # (N,S) bool
    set_names: list[str],          # length S
    true_labels: np.ndarray,       # length N
    normalize: str = "true",       # None | 'true' | 'pred' | 'all'
    min_support: int = 25,
    topk_pairs: int = 15,
    confident_mask: np.ndarray | None = None,
):
    N, S = front0_mask.shape
    assert len(set_names) == S and len(true_labels) == N

    # 1) filter to single-front rows (+ optional confidence)
    single = (front0_mask.sum(axis=1) == 1)
    ok = single & pd.notna(true_labels)
    if confident_mask is not None:
        ok &= confident_mask.astype(bool)
    if ok.sum() == 0:
        raise ValueError("No usable cells after filtering.")

    y_true = np.asarray(true_labels, dtype=object)[ok]
    y_pred = preds_from_single_front(front0_mask[ok], set_names)

    # 2) supports from ground truth
    support_counts = pd.Series(y_true).value_counts().sort_values(ascending=False)

    # 3) class order: majors then minors
    major_classes = support_counts[support_counts >= min_support].index.tolist()
    minor_classes = support_counts[support_counts <  min_support].index.tolist()
    ordered = major_classes + minor_classes

    # 4) confusions
    cm_counts = confusion_matrix(y_true, y_pred, labels=ordered, normalize=None)
    cm_norm   = confusion_matrix(y_true, y_pred, labels=ordered, normalize=normalize)
    df_counts = pd.DataFrame(cm_counts, index=ordered, columns=ordered)
    df_norm   = pd.DataFrame(cm_norm,   index=ordered, columns=ordered)

    # 5) per-class metrics table (drop report's support, merge our counts as 'support')
    rep = classification_report(y_true, y_pred, labels=ordered, output_dict=True, zero_division=0)
    per_class = (pd.DataFrame(rep).T
                 .rename_axis("class")
                 .reset_index())
    per_class = per_class[per_class["class"].isin(ordered)]
    if "support" in per_class.columns:
        per_class = per_class.drop(columns=["support"])
    per_class = (per_class
                 .merge(support_counts.rename("support").to_frame(),
                        left_on="class", right_index=True, how="left")
                 .fillna({"support": 0})
                 .sort_values("support", ascending=False))

    # 6) top error flows from normalized matrix (off-diagonals only)
    err = df_norm.copy()
    np.fill_diagonal(err.values, 0.0)
    pairs = []
    for i, ti in enumerate(err.index):
        row = err.iloc[i]
        jj = row.sort_values(ascending=False).index[:min(topk_pairs, len(row)-1)]
        for tj in jj:
            val = float(row[tj])
            if val > 0:
                pairs.append((ti, tj, val, int(df_counts.loc[ti, tj])))
    pairs = sorted(pairs, key=lambda x: (-x[2], -x[3]))[:topk_pairs]
    pairs_df = pd.DataFrame(pairs, columns=["true", "pred", f"norm({normalize})", "count"])

    # 7) plot majors-only normalized heatmap for readability
    majors = major_classes
    df_major = df_norm.loc[majors, majors] if majors else df_norm.iloc[[] , []]
    fig, ax = plt.subplots(figsize=(0.45*max(5,len(majors))+3, 0.45*max(5,len(majors))+3))
    im = ax.imshow(df_major.values, aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(len(majors))); ax.set_yticks(range(len(majors)))
    ax.set_xticklabels(majors, rotation=90); ax.set_yticklabels(majors)
    ax.set_title(f"Confusion (single-front; normalized={normalize})\n"
                 f"Used cells: {ok.sum()}  |  Shown true classes (≥{min_support}): {len(majors)}")
    # annotate diagonals
    for i in range(len(majors)):
        val = df_major.values[i,i]
        ax.text(i, i, f"{val:.2f}", ha="center", va="center",
                color="white" if val>0.5 else "black", fontsize=8)
    plt.tight_layout()

    return {
        "cm_counts": df_counts,
        "cm_norm": df_norm,
        "per_class": per_class,
        "top_errors": pairs_df,
        "idx_used": np.flatnonzero(ok),
        "figure": fig,
    }


In [ ]:
# Optional: confidence filter (keeps only crisp cells)
# margin_conf = Z_arr.max(1) - np.partition(Z_arr, -2, axis=1)[:, -2]
# confident = (Z_arr.max(1) >= 1.5) & (margin_conf >= 0.3)
confident = None  # or the boolean mask above

out = confusion_panels_single_front(
    front0_mask=front0_mask,
    set_names=set_names,
    true_labels=AD.obs['Type'],     # e.g., adata.obs["cell_type"]
    normalize="true",            # per-row recall
    min_support=25,              # hide tiny true classes in the main heatmap
    topk_pairs=15,
    confident_mask=confident
)

print(out["per_class"].round(3))   # precision/recall/f1/support table
print("\nTop error flows:\n", out["top_errors"])
# The plotted figure shows majors only; counts/normalized DFs include all classes.


In [ ]:
X_2d=umap.UMAP(verbose=True).fit_transform(Z)

In [ ]:
for T in AD.obs['Type'].unique():
    M=AD.obs['Type']==T
    plt.figure()
    plt.scatter(X_2d[:,0],X_2d[:,1],s=.1,color='gray')
    plt.scatter(X_2d[M,0],X_2d[M,1],s=1,label=f"{T}")
    plt.legend(markerscale=5,bbox_to_anchor=(1,1))
    plt.show()